# الجلسة 4 – مقارنة بين SLM و LLM

قارن بين زمن الاستجابة وجودة العينات الناتجة بين نموذج لغة صغير ونموذج أكبر يعمل عبر Foundry Local.


## ⚡ البداية السريعة

**إعداد مُحسّن للذاكرة (مُحدّث):**
1. النماذج تختار تلقائيًا إصدارات CPU (تعمل على أي جهاز)
2. يستخدم `qwen2.5-3b` بدلاً من 7B (يوفر حوالي 4 جيجابايت من الذاكرة)
3. الكشف التلقائي عن المنافذ (لا حاجة للإعداد اليدوي)
4. إجمالي الذاكرة المطلوبة: يُوصى بـ ~8 جيجابايت (النماذج + نظام التشغيل)

**إعداد الطرفية (30 ثانية):**
```bash
foundry service start
foundry model run phi-4-mini
foundry model run qwen2.5-3b
```

ثم قم بتشغيل هذا الدفتر! 🚀


### شرح: تثبيت التبعيات
يتم تثبيت الحد الأدنى من الحزم (`foundry-local-sdk`, `openai`, `numpy`) اللازمة لطلبات التوقيت والدردشة. يمكن إعادة تشغيله بأمان دون تأثير متكرر.


# السيناريو
قارن بين نموذج لغة صغير (SLM) ونموذج أكبر باستخدام موجه واحد لتوضيح المزايا والعيوب:
- **فرق زمن الاستجابة** (بالثواني الفعلية)
- **استخدام الرموز** (إذا كان متاحًا) كمؤشر على الإنتاجية
- **عينة من المخرجات النوعية** لتقييم سريع
- **حساب التسريع** لتحديد مكاسب الأداء

**المتغيرات البيئية:**
- `SLM_ALIAS` - نموذج اللغة الصغير (الإعداد الافتراضي: phi-4-mini، ~4GB RAM)
- `LLM_ALIAS` - نموذج اللغة الأكبر (الإعداد الافتراضي: qwen2.5-7b، ~7GB RAM)
- `COMPARE_PROMPT` - الموجه المستخدم للمقارنة
- `COMPARE_RETRIES` - عدد محاولات الإعادة لضمان الاستقرار (الإعداد الافتراضي: 2)
- `FOUNDRY_LOCAL_ENDPOINT` - تجاوز نقطة نهاية الخدمة (يتم اكتشافها تلقائيًا إذا لم يتم تحديدها)

**كيف يعمل (نمط SDK الرسمي):**
1. يقوم **FoundryLocalManager** بتهيئة وإدارة خدمة Foundry Local
2. يتم تشغيل الخدمة تلقائيًا إذا لم تكن قيد التشغيل (لا حاجة لإعداد يدوي)
3. يتم تحويل أسماء النماذج إلى معرفات محددة تلقائيًا
4. يتم اختيار النسخ المحسّنة للأجهزة (CUDA، NPU، أو CPU)
5. يقوم العميل المتوافق مع OpenAI بتنفيذ إكمال المحادثات
6. يتم جمع المقاييس: زمن الاستجابة، الرموز، جودة المخرجات
7. يتم مقارنة النتائج لحساب نسبة التسريع

هذه المقارنة المصغرة تساعد في اتخاذ القرار بشأن متى يكون من المبرر استخدام نموذج أكبر لحالتك.

**مرجع SDK:** 
- Python SDK: https://github.com/microsoft/Foundry-Local/tree/main/sdk/python/foundry_local
- أدوات الورشة: تستخدم النمط الرسمي من ../samples/workshop_utils.py

**الفوائد الرئيسية:**
- ✅ اكتشاف الخدمة وتهيئتها تلقائيًا
- ✅ تشغيل الخدمة تلقائيًا إذا لم تكن قيد التشغيل
- ✅ حل النماذج وتخزينها مؤقتًا تلقائيًا
- ✅ تحسين الأجهزة (CUDA/NPU/CPU)
- ✅ توافق مع SDK الخاص بـ OpenAI
- ✅ معالجة الأخطاء بشكل قوي مع محاولات الإعادة
- ✅ استنتاج محلي (لا حاجة إلى واجهات برمجة تطبيقات السحابة)


## 🚨 المتطلبات الأساسية: يجب تشغيل Foundry Local!

**قبل تشغيل هذا الدفتر**، تأكد من إعداد خدمة Foundry Local:

### أوامر البدء السريع (تشغيل في الطرفية):

```bash
# 1. Start the Foundry Local service
foundry service start

# 2. Load the default models used in this comparison (CPU-optimized)
foundry model run phi-4-mini
foundry model run qwen2.5-3b

# 3. Verify models are loaded
foundry model ls

# 4. Check service health
foundry service status
```

### نماذج بديلة (إذا لم تكن الإعدادات الافتراضية متوفرة):

```bash
# Even smaller alternatives (if memory is very limited)
foundry model run phi-3.5-mini
foundry model run qwen2.5-0.5b

# Or update the environment variables in this notebook:
# SLM_ALIAS = 'phi-3.5-mini'
# LLM_ALIAS = 'qwen2.5-1.5b'  # Or qwen2.5-0.5b for minimum memory
```

⚠️ **إذا قمت بتجاوز هذه الخطوات**، ستظهر لك رسالة `APIConnectionError` عند تشغيل خلايا الدفتر أدناه.


In [29]:
# Install dependencies
!pip install -q foundry-local-sdk openai numpy requests

### شرح: الواردات الأساسية
يجلب أدوات التوقيت وعملاء Foundry Local / OpenAI المستخدمين لجلب معلومات النموذج وتنفيذ إكمال المحادثات.


In [30]:
import os, time, json
from foundry_local import FoundryLocalManager
from openai import OpenAI
import sys
sys.path.append('../samples')
from workshop_utils import get_client, chat_once

### شرح: الأسماء المستعارة وإعداد المطالبات  
يحدد أسماء مستعارة قابلة للتكوين حسب البيئة للنموذج الصغير مقابل النموذج الأكبر بالإضافة إلى مطالبة مقارنة. قم بتعديل متغيرات البيئة لتجربة عائلات نماذج أو مهام مختلفة.


In [31]:
# Default to CPU models for better memory efficiency
SLM = os.getenv('SLM_ALIAS', 'phi-4-mini')  # Auto-selects CPU variant
LLM = os.getenv('LLM_ALIAS', 'qwen2.5-3b')  # Smaller LLM, more memory-friendly
PROMPT = os.getenv('COMPARE_PROMPT', 'List 5 benefits of local AI inference.')
# Endpoint is now managed by FoundryLocalManager - it auto-detects or can be overridden
ENDPOINT = os.getenv('FOUNDRY_LOCAL_ENDPOINT', None)

### 💡 إعداد مُحسّن للذاكرة

**يستخدم هذا الدفتر نماذج فعّالة من حيث استهلاك الذاكرة بشكل افتراضي:**
- `phi-4-mini` → ~4GB RAM (يختار Foundry Local تلقائيًا إصدار CPU)
- `qwen2.5-3b` → ~3GB RAM (بدلاً من 7B الذي يحتاج إلى ~7GB+)

**الكشف التلقائي عن المنافذ:**
- قد يستخدم Foundry Local منافذ مختلفة (غالبًا 55769 أو 59959)
- الخلية التشخيصية أدناه تكتشف المنفذ الصحيح تلقائيًا
- لا حاجة لإعداد يدوي!

**إذا كانت لديك ذاكرة محدودة (<8GB)، استخدم نماذج أصغر:**
```python
SLM = 'phi-3.5-mini'      # ~2GB
LLM = 'qwen2.5-0.5b'      # ~500MB
```


In [32]:
# Display current configuration
print("="*60)
print("CURRENT CONFIGURATION")
print("="*60)
print(f"SLM Model:     {SLM}")
print(f"LLM Model:     {LLM}")
print(f"SDK Pattern:   FoundryLocalManager (official)")
print(f"Endpoint:      {ENDPOINT or 'Auto-detect'}")
print(f"Test Prompt:   {PROMPT[:50]}...")
print(f"Retry Count:   2")
print("="*60)
print("\n💡 Using official Foundry SDK pattern from workshop_utils")
print("   → FoundryLocalManager handles service lifecycle")
print("   → Automatic model resolution and hardware optimization")
print("   → OpenAI-compatible API for inference")

CURRENT CONFIGURATION
SLM Model:     phi-4-mini
LLM Model:     qwen2.5-7b
SDK Pattern:   FoundryLocalManager (official)
Endpoint:      Auto-detect
Test Prompt:   List 5 benefits of local AI inference....
Retry Count:   2

💡 Using official Foundry SDK pattern from workshop_utils
   → FoundryLocalManager handles service lifecycle
   → Automatic model resolution and hardware optimization
   → OpenAI-compatible API for inference


### شرح: أدوات تنفيذ (نمط Foundry SDK)
يستخدم النمط الرسمي لـ Foundry Local SDK كما هو موثق في عينات ورش العمل:

**النهج:**
- **FoundryLocalManager** - يقوم بتهيئة وإدارة خدمة Foundry Local
- **الاكتشاف التلقائي** - يكتشف النقاط النهائية تلقائيًا ويتعامل مع دورة حياة الخدمة
- **حل النماذج** - يحل الأسماء المستعارة إلى معرفات النماذج الكاملة (مثل phi-4-mini → phi-4-mini-instruct-cpu)
- **تحسين الأجهزة** - يختار أفضل إصدار للأجهزة المتاحة (CUDA، NPU، أو CPU)
- **عميل OpenAI** - يتم تكوينه باستخدام نقطة النهاية الخاصة بالمدير للوصول إلى واجهة برمجة التطبيقات المتوافقة مع OpenAI

**ميزات المرونة:**
- منطق إعادة المحاولة مع زيادة تدريجية (قابل للتكوين عبر البيئة)
- بدء تشغيل الخدمة تلقائيًا إذا لم تكن قيد التشغيل
- التحقق من الاتصال بعد التهيئة
- التعامل مع الأخطاء بسلاسة مع تقديم تقارير تفصيلية عن الأخطاء
- تخزين النماذج مؤقتًا لتجنب التهيئة المتكررة

**هيكل النتائج:**
- قياس زمن التأخير (وقت الساعة الحائطية)
- تتبع استخدام الرموز (إذا كان متاحًا)
- عينة من المخرجات (مختصرة لسهولة القراءة)
- تفاصيل الأخطاء للطلبات الفاشلة

يعتمد هذا النمط على وحدة workshop_utils التي تتبع النمط الرسمي لـ SDK.

**مرجع SDK:**
- المستودع الرئيسي: https://github.com/microsoft/Foundry-Local
- Python SDK: https://github.com/microsoft/Foundry-Local/tree/main/sdk/python/foundry_local
- أدوات ورشة العمل: ../samples/workshop_utils.py


In [39]:
def setup(alias: str, endpoint: str = None, retries: int = 3):
    """
    Initialize a Foundry Local model connection using official SDK pattern.
    
    This follows the workshop_utils pattern which uses FoundryLocalManager
    to properly initialize the Foundry Local service and resolve models.
    
    Args:
        alias: Model alias (e.g., 'phi-4-mini', 'qwen2.5-3b')
        endpoint: Optional endpoint override (usually auto-detected)
        retries: Number of connection attempts (default: 3)
    
    Returns:
        tuple: (manager, client, model_id, metadata) or (None, None, alias, error_metadata) if failed
    """
    import time
    
    last_err = None
    current_delay = 2  # seconds
    
    for attempt in range(1, retries + 1):
        try:
            print(f"[Init] Connecting to '{alias}' (attempt {attempt}/{retries})...")
            
            # Use the workshop utility which follows the official SDK pattern
            manager, client, model_id = get_client(alias, endpoint=endpoint)
            
            print(f"[OK] Connected to '{alias}' -> {model_id}")
            print(f"     Endpoint: {manager.endpoint}")
            
            return manager, client, model_id, {
                'endpoint': manager.endpoint,
                'resolved': model_id,
                'attempts': attempt,
                'status': 'success'
            }
            
        except Exception as e:
            last_err = e
            error_msg = str(e)
            
            # Provide helpful error messages
            if "Connection error" in error_msg or "connection refused" in error_msg.lower():
                print(f"[ERROR] Cannot connect to Foundry Local service")
                print(f"        → Is the service running? Try: foundry service start")
                print(f"        → Is the model loaded? Try: foundry model run {alias}")
            elif "not found" in error_msg.lower():
                print(f"[ERROR] Model '{alias}' not found in catalog")
                print(f"        → Available models: Run 'foundry model ls' in terminal")
                print(f"        → Download model: Run 'foundry model download {alias}'")
            else:
                print(f"[ERROR] Setup failed: {type(e).__name__}: {error_msg}")
            
            if attempt < retries:
                print(f"[Retry] Waiting {current_delay:.1f}s before retry...")
                time.sleep(current_delay)
                current_delay *= 2  # Exponential backoff
    
    # All retries failed - provide actionable guidance
    print(f"\n❌ Failed to initialize '{alias}' after {retries} attempts")
    print(f"   Last error: {type(last_err).__name__}: {str(last_err)}")
    print(f"\n💡 Troubleshooting steps:")
    print(f"   1. Ensure Foundry Local service is running:")
    print(f"      → foundry service status")
    print(f"      → foundry service start (if not running)")
    print(f"   2. Ensure model is loaded:")
    print(f"      → foundry model run {alias}")
    print(f"   3. Check available models:")
    print(f"      → foundry model ls")
    print(f"   4. Try alternative models if '{alias}' isn't available")
    
    return None, None, alias, {
        'error': f"{type(last_err).__name__}: {str(last_err)}",
        'endpoint': endpoint or 'auto-detect',
        'attempts': retries,
        'status': 'failed'
    }


def run(client, model_id: str, prompt: str, max_tokens: int = 180, temperature: float = 0.5):
    """
    Run inference with the configured model using OpenAI SDK.
    
    Args:
        client: OpenAI client instance (configured for Foundry Local)
        model_id: Model identifier (resolved from alias)
        prompt: Input prompt
        max_tokens: Maximum response tokens (default: 180)
        temperature: Sampling temperature (default: 0.5)
    
    Returns:
        dict: Response with timing, tokens, and content
    """
    import time
    
    start = time.time()
    
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature
        )
        
        elapsed = time.time() - start
        
        # Extract response details
        content = response.choices[0].message.content
        
        # Try to extract token usage from multiple possible locations
        usage_info = {}
        if hasattr(response, 'usage') and response.usage:
            usage_info['prompt_tokens'] = getattr(response.usage, 'prompt_tokens', None)
            usage_info['completion_tokens'] = getattr(response.usage, 'completion_tokens', None)
            usage_info['total_tokens'] = getattr(response.usage, 'total_tokens', None)
        
        # Calculate approximate token count if API doesn't provide it
        # Rough estimate: ~4 characters per token for English text
        if not usage_info.get('total_tokens'):
            estimated_prompt_tokens = len(prompt) // 4
            estimated_completion_tokens = len(content) // 4
            estimated_total = estimated_prompt_tokens + estimated_completion_tokens
            usage_info['estimated_tokens'] = estimated_total
            usage_info['estimated_prompt_tokens'] = estimated_prompt_tokens
            usage_info['estimated_completion_tokens'] = estimated_completion_tokens
        
        return {
            'status': 'success',
            'content': content,
            'elapsed_sec': elapsed,
            'tokens': usage_info.get('total_tokens') or usage_info.get('estimated_tokens'),
            'usage': usage_info,
            'model': model_id
        }
        
    except Exception as e:
        elapsed = time.time() - start
        return {
            'status': 'error',
            'error': f"{type(e).__name__}: {str(e)}",
            'elapsed_sec': elapsed,
            'model': model_id
        }


print("✅ Execution helpers defined: setup(), run()")
print("   → Uses workshop_utils for proper SDK integration")
print("   → setup() initializes with FoundryLocalManager")
print("   → run() executes inference via OpenAI-compatible API")
print("   → Token counting: Uses API data or estimates if unavailable")

✅ Execution helpers defined: setup(), run()
   → Uses workshop_utils for proper SDK integration
   → setup() initializes with FoundryLocalManager
   → run() executes inference via OpenAI-compatible API
   → Token counting: Uses API data or estimates if unavailable


### شرح: اختبار ذاتي قبل التشغيل
يجري فحص اتصال خفيف باستخدام FoundryLocalManager لكلا النموذجين. هذا الفحص يتحقق من:
- إمكانية الوصول إلى الخدمة
- إمكانية تهيئة النماذج
- حل الأسماء المستعارة إلى معرفات النماذج الفعلية
- استقرار الاتصال قبل إجراء المقارنة

تستخدم وظيفة setup() النمط الرسمي لـ SDK من workshop_utils.


In [34]:
# Simplified diagnostic: Just verify service is accessible
import requests

def check_foundry_service():
    """Quick diagnostic to verify Foundry Local is running."""
    # Try common ports
    endpoints_to_try = [
        "http://localhost:59959",
        "http://127.0.0.1:59959", 
        "http://localhost:55769",
        "http://127.0.0.1:55769",
    ]
    
    print("[Diagnostic] Checking Foundry Local service...")
    
    for endpoint in endpoints_to_try:
        try:
            response = requests.get(f"{endpoint}/health", timeout=2)
            if response.status_code == 200:
                print(f"✅ Service is running at {endpoint}")
                
                # Try to list models
                try:
                    models_response = requests.get(f"{endpoint}/v1/models", timeout=2)
                    if models_response.status_code == 200:
                        models_data = models_response.json()
                        model_count = len(models_data.get('data', []))
                        print(f"✅ Found {model_count} models available")
                        if model_count > 0:
                            print("   Models:", [m.get('id', 'unknown') for m in models_data.get('data', [])[:5]])
                except Exception as e:
                    print(f"⚠️  Could not list models: {e}")
                
                return endpoint
        except requests.exceptions.ConnectionError:
            continue
        except Exception as e:
            print(f"⚠️  Error checking {endpoint}: {e}")
    
    print("\n❌ Foundry Local service not found!")
    print("\n💡 To fix this:")
    print("   1. Open a terminal")
    print("   2. Run: foundry service start")
    print("   3. Run: foundry model run phi-4-mini")
    print("   4. Run: foundry model run qwen2.5-3b")
    print("   5. Re-run this notebook")
    return None

# Run diagnostic
discovered_endpoint = check_foundry_service()

if discovered_endpoint:
    print(f"\n✅ Service detected (will be managed by FoundryLocalManager)")
else:
    print(f"\n⚠️  No service detected - FoundryLocalManager will attempt to start it")

[Diagnostic] Checking Foundry Local service...

❌ Foundry Local service not found!

💡 To fix this:
   1. Open a terminal
   2. Run: foundry service start
   3. Run: foundry model run phi-4-mini
   4. Run: foundry model run qwen2.5-3b
   5. Re-run this notebook

⚠️  No service detected - FoundryLocalManager will attempt to start it


In [35]:
# Quick Fix: Start service and load models from notebook
# Uncomment the commands you need:

# !foundry service start
# !foundry model run phi-4-mini
# !foundry model run qwen2.5-3b
# !foundry model ls

print("⚠️  The commands above are commented out.")
print("Uncomment them if you want to start the service from the notebook.")
print("")
print("💡 Recommended: Run these commands in a separate terminal instead:")
print("   foundry service start")
print("   foundry model run phi-4-mini")
print("   foundry model run qwen2.5-3b")

⚠️  The commands above are commented out.
Uncomment them if you want to start the service from the notebook.

💡 Recommended: Run these commands in a separate terminal instead:
   foundry service start
   foundry model run phi-4-mini
   foundry model run qwen2.5-3b


### 🛠️ إصلاح سريع: بدء تشغيل Foundry Local من دفتر الملاحظات (اختياري)

إذا أظهر التشخيص أعلاه أن الخدمة لا تعمل، يمكنك محاولة تشغيلها من هنا:

**ملاحظة:** هذا يعمل بشكل أفضل على نظام Windows. على الأنظمة الأخرى، استخدم أوامر الطرفية.


### ⚠️ استكشاف أخطاء اتصال الخدمة وإصلاحها

إذا كنت ترى `APIConnectionError`، فقد تكون خدمة Foundry Local غير قيد التشغيل أو النماذج غير محملة. جرب الخطوات التالية:

**1. تحقق من حالة الخدمة:**
```bash
# In a terminal (not in notebook):
foundry service status
```

**2. قم بتشغيل الخدمة (إذا لم تكن قيد التشغيل):**
```bash
foundry service start
```

**3. قم بتحميل النماذج المطلوبة:**
```bash
# Load the models needed for comparison
foundry model run phi-4-mini
foundry model run qwen2.5-7b

# Or use alternative models:
foundry model run phi-3.5-mini
foundry model run qwen2.5-3b
```

**4. تحقق من توفر النماذج:**
```bash
foundry model ls
```

**المشاكل الشائعة:**
- ❌ الخدمة غير قيد التشغيل → قم بتشغيل `foundry service start`
- ❌ النماذج غير محملة → قم بتشغيل `foundry model run <model-name>`
- ❌ تعارض المنافذ → تحقق مما إذا كانت خدمة أخرى تستخدم المنفذ
- ❌ جدار الحماية يحظر → تأكد من السماح بالاتصالات المحلية

**إصلاح سريع:** قم بتشغيل خلية التشخيص أدناه قبل التحقق المبدئي.


In [36]:
preflight = {}
retries = 2  # Number of retry attempts

for a in (SLM, LLM):
    mgr, c, mid, info = setup(a, endpoint=ENDPOINT, retries=retries)
    # Keep the original status from info (either 'success' or 'failed')
    preflight[a] = info

print('\n[Pre-flight Check]')
for alias, details in preflight.items():
    status_icon = '✅' if details['status'] == 'success' else '❌'
    print(f"  {status_icon} {alias}: {details['status']} - {details.get('resolved', details.get('error', 'unknown'))}")

preflight

[Init] Connecting to 'phi-4-mini' (attempt 1/2)...
[OK] Connected to 'phi-4-mini' -> Phi-4-mini-instruct-cuda-gpu:4
     Endpoint: http://127.0.0.1:59959/v1
[Init] Connecting to 'qwen2.5-7b' (attempt 1/2)...
[OK] Connected to 'qwen2.5-7b' -> qwen2.5-7b-instruct-cuda-gpu:3
     Endpoint: http://127.0.0.1:59959/v1

[Pre-flight Check]
  ✅ phi-4-mini: success - Phi-4-mini-instruct-cuda-gpu:4
  ✅ qwen2.5-7b: success - qwen2.5-7b-instruct-cuda-gpu:3


{'phi-4-mini': {'endpoint': 'http://127.0.0.1:59959/v1',
  'resolved': 'Phi-4-mini-instruct-cuda-gpu:4',
  'attempts': 1,
  'status': 'success'},
 'qwen2.5-7b': {'endpoint': 'http://127.0.0.1:59959/v1',
  'resolved': 'qwen2.5-7b-instruct-cuda-gpu:3',
  'attempts': 1,
  'status': 'success'}}

### ✅ التحقق المسبق: توفر النموذج

تتحقق هذه الخلية من إمكانية الوصول إلى كلا النموذجين عند نقطة النهاية المُعدة قبل تشغيل المقارنة.


### شرح: مقارنة التشغيل وجمع النتائج
يقوم بالتكرار على كلا الاسمين المستعارين باستخدام نمط SDK الرسمي لـ Foundry:
1. تهيئة كل نموذج باستخدام setup() (يستخدم FoundryLocalManager)
2. تشغيل الاستدلال باستخدام واجهة برمجة التطبيقات المتوافقة مع OpenAI
3. تسجيل زمن الاستجابة، الرموز، وعينة من المخرجات
4. إنتاج ملخص JSON مع تحليل مقارن

هذا يتبع نفس النمط الموجود في عينات الورشة في session04/model_compare.py.


In [40]:
results = []
retries = 2  # Number of retry attempts

for alias in (SLM, LLM):
    mgr, client, mid, info = setup(alias, endpoint=ENDPOINT, retries=retries)
    if client:
        r = run(client, mid, PROMPT)
        results.append({'alias': alias, **r})
    else:
        # If setup failed, record error
        results.append({
            'alias': alias,
            'status': 'error',
            'error': info.get('error', 'Setup failed'),
            'elapsed_sec': 0,
            'tokens': None,
            'model': alias
        })

# Display results
print(json.dumps(results, indent=2))

# Quick comparative view
print('\n' + '='*80)
print('COMPARISON SUMMARY')
print('='*80)
print(f"{'Alias':<20} {'Status':<15} {'Latency(s)':<15} {'Tokens':<15}")
print('-'*80)

for row in results:
    status = row.get('status', 'unknown')
    status_icon = '✅' if status == 'success' else '❌'
    latency_str = f"{row.get('elapsed_sec', 0):.3f}" if row.get('elapsed_sec') else 'N/A'
    
    # Handle token display - show if available or indicate estimated
    tokens = row.get('tokens')
    usage = row.get('usage', {})
    if tokens:
        if 'estimated_tokens' in usage:
            tokens_str = f"~{tokens} (est.)"
        else:
            tokens_str = str(tokens)
    else:
        tokens_str = 'N/A'
    
    print(f"{status_icon} {row['alias']:<18} {status:<15} {latency_str:<15} {tokens_str:<15}")

print('-'*80)

# Show detailed token breakdown if available
print("\nDetailed Token Usage:")
for row in results:
    if row.get('status') == 'success' and row.get('usage'):
        usage = row['usage']
        print(f"\n  {row['alias']}:")
        if 'prompt_tokens' in usage and usage['prompt_tokens']:
            print(f"    Prompt tokens:     {usage['prompt_tokens']}")
            print(f"    Completion tokens: {usage['completion_tokens']}")
            print(f"    Total tokens:      {usage['total_tokens']}")
        elif 'estimated_tokens' in usage:
            print(f"    Estimated prompt:     {usage['estimated_prompt_tokens']}")
            print(f"    Estimated completion: {usage['estimated_completion_tokens']}")
            print(f"    Estimated total:      {usage['estimated_tokens']}")
            print(f"    (API did not provide token counts - using ~4 chars/token estimate)")

print('\n' + '='*80)

# Calculate speedup if both succeeded
if len(results) == 2 and all(r.get('status') == 'success' and r.get('elapsed_sec') for r in results):
    speedup = results[1]['elapsed_sec'] / results[0]['elapsed_sec']
    print(f"\n💡 SLM is {speedup:.2f}x faster than LLM for this prompt")
    
    # Compare token throughput if available
    slm_tokens = results[0].get('tokens', 0)
    llm_tokens = results[1].get('tokens', 0)
    if slm_tokens and llm_tokens:
        slm_tps = slm_tokens / results[0]['elapsed_sec']
        llm_tps = llm_tokens / results[1]['elapsed_sec']
        print(f"   SLM throughput: {slm_tps:.1f} tokens/sec")
        print(f"   LLM throughput: {llm_tps:.1f} tokens/sec")
        
elif any(r.get('status') == 'error' for r in results):
    print(f"\n⚠️  Some models failed - check errors above")
    print("   Ensure Foundry Local is running: foundry service start")
    print("   Ensure models are loaded: foundry model run <model-name>")

results

[Init] Connecting to 'phi-4-mini' (attempt 1/2)...
[OK] Connected to 'phi-4-mini' -> Phi-4-mini-instruct-cuda-gpu:4
     Endpoint: http://127.0.0.1:59959/v1
[Init] Connecting to 'qwen2.5-7b' (attempt 1/2)...
[OK] Connected to 'qwen2.5-7b' -> qwen2.5-7b-instruct-cuda-gpu:3
     Endpoint: http://127.0.0.1:59959/v1
[Init] Connecting to 'qwen2.5-7b' (attempt 1/2)...
[OK] Connected to 'qwen2.5-7b' -> qwen2.5-7b-instruct-cuda-gpu:3
     Endpoint: http://127.0.0.1:59959/v1
[
  {
    "alias": "phi-4-mini",
    "status": "success",
    "content": "1. Reduced Latency: Local AI inference can significantly reduce latency by processing data closer to the source, which is particularly beneficial for real-time applications such as autonomous vehicles or augmented reality.\n\n2. Enhanced Privacy: By keeping data processing local, sensitive information is less likely to be exposed to external networks, thereby enhancing privacy and security.\n\n3. Lower Bandwidth Usage: Local AI inference reduces the n

[{'alias': 'phi-4-mini',
  'status': 'success',
  'content': '1. Reduced Latency: Local AI inference can significantly reduce latency by processing data closer to the source, which is particularly beneficial for real-time applications such as autonomous vehicles or augmented reality.\n\n2. Enhanced Privacy: By keeping data processing local, sensitive information is less likely to be exposed to external networks, thereby enhancing privacy and security.\n\n3. Lower Bandwidth Usage: Local AI inference reduces the need for data transmission over the network, which can save bandwidth and reduce the risk of network congestion.\n\n4. Improved Reliability: Local processing can be more reliable, as it is less dependent on network connectivity. This is particularly important in scenarios where network connectivity is unreliable or intermittent.\n\n5. Scalability: Local AI inference can be easily scaled by adding more local processing units, making it easier to handle increasing data volumes or m

### تفسير النتائج

**المقاييس الرئيسية:**
- **زمن الاستجابة**: كلما كان أقل، كان أفضل - يشير إلى سرعة الاستجابة
- **الرموز**: معدل أعلى = معالجة المزيد من الرموز
- **المسار**: يؤكد أي نقطة نهاية API تم استخدامها

**متى تستخدم SLM مقابل LLM:**
- **SLM (نموذج اللغة الصغيرة)**: استجابات سريعة، استخدام أقل للموارد، مناسب للمهام البسيطة
- **LLM (نموذج اللغة الكبير)**: جودة أعلى، تفكير أفضل، يُستخدم عندما تكون الجودة هي الأهم

**الخطوات التالية:**
1. جرب مطالبات مختلفة لترى كيف تؤثر التعقيدات على المقارنة
2. قم بتجربة أزواج نماذج أخرى
3. استخدم عينات التوجيه من ورشة العمل (الجلسة 06) لتوجيه المهام بذكاء بناءً على تعقيدها


In [38]:
# Final Validation Check
print("="*70)
print("VALIDATION SUMMARY")
print("="*70)
print(f"✅ SLM Model: {SLM}")
print(f"✅ LLM Model: {LLM}")
print(f"✅ Using Foundry SDK Pattern: workshop_utils with FoundryLocalManager")
print(f"✅ Pre-flight passed: {all(v['status'] == 'success' for v in preflight.values()) if 'preflight' in dir() else 'Not run yet'}")
print(f"✅ Comparison completed: {len(results) == 2 if 'results' in dir() else 'Not run yet'}")
print(f"✅ Both models responded: {all(r.get('status') == 'success' for r in results) if 'results' in dir() and results else 'Not run yet'}")
print("="*70)

# Check for common configuration issues
issues = []
if 'LLM' in dir() and LLM not in ['qwen2.5-3b', 'qwen2.5-0.5b', 'qwen2.5-1.5b', 'qwen2.5-7b', 'phi-3.5-mini']:
    issues.append(f"⚠️  LLM is '{LLM}' - expected qwen2.5-3b for memory efficiency")
if 'preflight' in dir() and not all(v['status'] == 'success' for v in preflight.values()):
    issues.append("⚠️  Pre-flight check failed - models not accessible")
if 'results' in dir() and results and not all(r.get('status') == 'success' for r in results):
    issues.append("⚠️  Comparison incomplete - check for errors above")

if not issues and 'results' in dir() and results and all(r.get('status') == 'success' for r in results):
    print("🎉 ALL CHECKS PASSED! Notebook completed successfully.")
    print(f"   SLM ({SLM}) vs LLM ({LLM}) comparison completed.")
    if len(results) == 2:
        speedup = results[1]['elapsed_sec'] / results[0]['elapsed_sec'] if results[0]['elapsed_sec'] > 0 else 0
        print(f"   Performance: SLM is {speedup:.2f}x faster")
elif issues:
    print("\n⚠️  Issues detected:")
    for issue in issues:
        print(f"   {issue}")
    print("\n💡 Troubleshooting:")
    print("   1. Ensure service is running: foundry service start")
    print("   2. Load models: foundry model run phi-4-mini && foundry model run qwen2.5-7b")
    print("   3. Check model list: foundry model ls")
else:
    print("\n💡 Run all cells above first, then re-run this validation.")
print("="*70)

VALIDATION SUMMARY
✅ SLM Model: phi-4-mini
✅ LLM Model: qwen2.5-7b
✅ Using Foundry SDK Pattern: workshop_utils with FoundryLocalManager
✅ Pre-flight passed: True
✅ Comparison completed: True
✅ Both models responded: True
🎉 ALL CHECKS PASSED! Notebook completed successfully.
   SLM (phi-4-mini) vs LLM (qwen2.5-7b) comparison completed.
   Performance: SLM is 5.14x faster



---

**إخلاء المسؤولية**:  
تم ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو معلومات غير دقيقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي. للحصول على معلومات حاسمة، يُوصى بالاستعانة بترجمة بشرية احترافية. نحن غير مسؤولين عن أي سوء فهم أو تفسيرات خاطئة ناتجة عن استخدام هذه الترجمة.
